In [ ]:
from torch import nn

class Encoder(nn.Module):
    """编码器-解码器架构的基础编码器接口
    
    编码器的作用是将输入序列（如一句话）转换为一个固定维度的上下文向量（context vector），
    这个向量包含了输入序列的所有重要信息。
    """
    def __init__(self, **kwargs):
        super(Encoder, self).__init__(**kwargs)

    def forward(self, X, *args):
        """编码一个批次的序列
        
        参数:
            X: 输入序列，通常是一个形状为 (batch_size, seq_length) 的张量
            *args: 其他可选参数，具体含义取决于具体的实现
            
        返回:
            编码后的输出，通常包含隐藏状态等信息
            
        注意:
            这是一个抽象方法，需要在子类中具体实现
        """
        raise NotImplementedError


In [ ]:
class Decoder(nn.Module):
    """编码器-解码器架构的基础解码器接口
    
    解码器的作用是根据编码器生成的上下文向量，逐步生成输出序列（如翻译后的句子）。
    解码过程通常是自回归的，即当前时间步的输出会作为下一个时间步的输入。
    """
    def __init__(self, **kwargs):
        super(Decoder, self).__init__(**kwargs)

    def init_state(self, enc_outputs, *args):
        """基于编码器的输出初始化解码器的状态（上下文向量）
        
        参数:
            enc_outputs: 编码器的输出，包含了输入序列的编码信息
            *args: 其他可选参数，具体含义取决于具体的实现
            
        返回:
            解码器的初始状态，通常包括：
            - 上下文向量（context vector）
            - 隐藏状态（hidden state）
            
        注意:
            这是一个抽象方法，需要在子类中具体实现
        """
        raise NotImplementedError

    def forward(self, X, state):
        """解码一个批次的序列
        
        参数:
            X: 解码器的输入序列，通常是目标序列的前缀
               形状为 (batch_size, seq_length)
            state: 解码器的当前状态，由 init_state 方法初始化
            
        返回:
            解码器的输出，通常是预测的词元概率分布
            
        注意:
            这是一个抽象方法，需要在子类中具体实现
        """
        raise NotImplementedError


In [ ]:
class EncoderDecoder(nn.Module):
    """编码器-解码器架构的基类
    
    这是一个通用的序列到序列（seq2seq）模型框架，广泛应用于：
    - 机器翻译：将一种语言的句子翻译成另一种语言
    - 文本摘要：将长文本压缩成简短摘要
    - 对话系统：根据输入生成回复
    
    工作流程：
    1. 编码器将输入序列编码成上下文向量
    2. 解码器根据上下文向量和目标序列前缀，生成输出序列
    """
    def __init__(self, encoder, decoder, **kwargs):
        """初始化编码器-解码器模型
        
        参数:
            encoder: 编码器实例，必须是 Encoder 类的子类
            decoder: 解码器实例，必须是 Decoder 类的子类
        """
        super(EncoderDecoder, self).__init__(**kwargs)
        self.encoder = encoder  # 保存编码器
        self.decoder = decoder  # 保存解码器

    def forward(self, enc_X, dec_X, *args):
        """前向传播：完整的编码-解码过程
        
        参数:
            enc_X: 编码器的输入序列（源序列），如英文句子
                   形状: (batch_size, src_seq_length)
            dec_X: 解码器的输入序列（目标序列的前缀），如法文句子的开头部分
                   形状: (batch_size, tgt_seq_length)
            *args: 其他可选参数
            
        返回:
            解码器的输出，通常是目标序列每个位置的词元预测
            
        执行流程：
            1. 使用编码器处理源序列 enc_X，得到编码输出 enc_outputs
            2. 使用编码输出初始化解码器状态 dec_state
            3. 解码器根据目标序列前缀 dec_X 和状态 dec_state 生成预测
        """
        # 步骤1: 编码源序列
        enc_outputs = self.encoder(enc_X, *args)
        
        # 步骤2: 初始化解码器状态
        dec_state = self.decoder.init_state(enc_outputs, *args)
        
        # 步骤3: 解码生成输出
        return self.decoder(dec_X, dec_state)
